# Adaptive Corrective Self-RAG (ACSRAG) Complete Interactive Demo

This notebook demonstrates the complete **Adaptive Corrective Self-RAG (ACSRAG)** pipeline combining:
1. **Adaptive Intent Routing**: Classifies queries into `FACTUAL`, `ANALYTICAL`, or `OPINION`.
2. **Hybrid Retrieval with Root Header Priority**: Combines dense vector similarity with BM25 keyword matching and root header anchor boosting for candidate/document identity.
3. **Corrective RAG (CRAG) Document Grader**: Strictly verifies internal document relevance first (`CORRECT`), routing to live Web Search (`INCORRECT (Web Fallback)`) only when the document lacks the concept and Web Search is active.
4. **Self-RAG Reflection & Claim Auditing**: Sentence-level claim extraction, support verification, and calibrated confidence scoring.
5. **Bounded Iterations**: Maximum iteration limit strictly capped at **2 iterations** to eliminate infinite loops.

In [ ]:
import os

# Set your API keys (or load from environment / .env.local)
os.environ['GOOGLE_API_KEY'] = os.environ.get('GOOGLE_API_KEY', '<YOUR_GOOGLE_API_KEY_HERE>')
os.environ['TAVILY_API_KEY'] = os.environ.get('TAVILY_API_KEY', '<YOUR_TAVILY_API_KEY_HERE>')
print('✅ Environment configured.')

In [ ]:
from pathlib import Path
from acsrag.graphs.phase8_iterative import build_phase8_graph

# Load all documents from the documents directory
pdf_paths = list(Path('documents').glob('*.pdf'))
if not pdf_paths:
    pdf_paths = list(Path('acsrag/documents').glob('*.pdf'))

print(f'Loading {len(pdf_paths)} documents:', [p.name for p in pdf_paths])
graph = build_phase8_graph(pdf_paths)
print('✅ ACSRAG Phase 8 Graph initialized successfully (bounded to max 2 iterations).')

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception as e:
    print('Mermaid graph visualization available in visual environment:', e)

## 🔄 Streaming Step-by-Step Node Execution (Max 2 Iterations)
Inspect every individual node transition, state payload, intent, generation snippet, and confidence score during the live stream.

In [ ]:
query = "According to NexaAI Solutions' leave policy, how many working days of sick leave and casual leave are employees entitled to per year, and how many days of unused annual leave can be carried forward?"

for event in graph.stream({'question': query, 'iterations': 0}, stream_mode='updates'):
    for node, state in event.items():
        print(f'\n--- Node: {node} ---')
        if state and 'generation' in state:
            print(f'Generation snippet: {state["generation"][:150]}...')
        if state and 'intent' in state:
            print(f'Intent: {state["intent"]}')
        if state and 'confidence_scores' in state:
            print(f'Scores: {state["confidence_scores"]}')

## 📊 ACSRAG Process Trace Visualizer
The helper function below summarizes the end-to-end trace metrics, claim audits, and confidence scores.

In [ ]:
def print_trace(state):
    print("\n" + "="*50)
    print("           ACSRAG PROCESS TRACE")
    print("="*50)
    print(f"USER QUERY: {state.get('question')}\n    ↓")
    if state.get("intent"):
        print(f"Intent Classified: {state['intent']}\n    ↓")
    if state.get("retrieval_query"):
        print(f"Effective Query: {state['retrieval_query']}\n    ↓")
        
    print(f"Vector matches: {len(state.get('docs', []))}")
    print(f"BM25 matches: {len(state.get('bm25_docs', []))}")
    print(f"RRF fused: {len(state.get('fused_docs', []))}\n    ↓")
    
    print(f"CRAG Verdict: {state.get('verdict', 'CORRECT')}")
    print(f"Relevant documents: {len(state.get('good_docs', []))}/{len(state.get('fused_docs', []))}\n    ↓")
    print(f"Context passages: {len(state.get('refined_context', '').split(chr(10)+chr(10))) if state.get('refined_context') else 0}\n    ↓")
    
    claims = state.get("claims", [])
    claim_verdicts = state.get("claim_verdicts", [])
    supported = sum(1 for v in claim_verdicts if v.get("status", "").upper() == "SUPPORTED")
    
    print(f"Supported Claims: {supported} / Unsupported: {len(claims) - supported}\n    ↓")
    print(f"RETRIEVAL REQUIRED: {'YES' if state.get('need_retrieval', True) else 'NO'}\n    ↓")
    
    scores = state.get('confidence_scores', {})
    overall = scores.get('overall_confidence', 'N/A')
    print(f"Overall Confidence: {overall}")
    print("="*50 + "\n")

## 🧪 Multi-Scenario Benchmark Suite
Running the bounded RAG pipeline (max 2 iterations) across diverse scenarios:
1. **Internal Policy Query**: Sick leave, casual leave, and annual leave carry-forward.
2. **Candidate Identity & Credentials**: Document Header Chunk 0 extraction.
3. **Conceptual Domain Comparison (Web Fallback)**: General concepts not in document (`LLM vs RAG`).
4. **Hybrid Trend Query**: Combining internal resume skills with live 2026 industry demand.

In [ ]:
queries = [
    # 1. Company Leave Policy Query
    "According to NexaAI Solutions' leave policy, how many working days of sick leave and casual leave are employees entitled to per year, and how many days of unused annual leave can be carried forward?",
    
    # 2. Candidate Identity from Header (Document-first)
    "What is the name, email address, and CGPA of the candidate in the resume?",
    
    # 3. General Concept (Triggers Web Search fallback when concept is not in resume)
    "LLM vs RAG",
    
    # 4. Hybrid Query (Combines Resume skills + Live Web Search)
    "Compare my AI/ML skills from the resume with current 2026 data science industry demand"
]

for i, query in enumerate(queries, 1):
    print(f"\n\n{'#'*60}")
    print(f"DEMO QUERY {i}/{len(queries)}: \"{query}\"")
    print(f"{'#'*60}")
    
    # Run pipeline bounded to max 2 iterations
    final_state = graph.invoke({'question': query, 'iterations': 0})
    print_trace(final_state)
    
    print("FINAL GENERATED ANSWER:")
    print(final_state.get('answer', 'No answer generated.'))